# Character reference (GitHub source + Kaggle GPU)

This notebook is the Kaggle launcher and is tracked in GitHub. It clones the public project repository and runs `pipeline/kaggle_character_reference.py` from that clone, so the model code and prompt are maintained in GitHub rather than copied into an independent Kaggle script. Push the latest notebook and script to the required public repository before using this notebook.

Keep the reference photo in a **private Kaggle dataset**, never in GitHub. Only use it if you have permission. It guides broad visual qualities for a new fictional character; the code explicitly avoids copying the person's identity. Add your Hugging Face access token as a Kaggle Secret named `HF_TOKEN`; never paste it into a notebook cell or commit it. The notebook runs one generation and does not retry automatically.

## 1. Install the pinned Kaggle-session dependencies

Enable Kaggle Internet and select a GPU accelerator. This installs packages in the temporary Kaggle session only; it does not edit your GitHub repository or laptop environment.

In [ ]:
%pip install -q "diffusers==0.35.1" "transformers==4.49.0" "accelerate==1.2.1" "safetensors==0.5.3" "jsonschema==4.23.0" "Pillow==11.1.0"

## 2. Verify the GPU and locate the private photo

Attach the private photo dataset in Kaggle's **Add Input** panel. This cell only lists image paths inside the Kaggle session; it does not upload the image to GitHub.

In [ ]:
from pathlib import Path
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before continuing.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=True)

image_files = sorted(
    path for path in Path("/kaggle/input").rglob("*")
    if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
)
if not image_files:
    raise FileNotFoundError("Attach the private reference-photo dataset under Add Input.")
for path in image_files:
    print(path)

In [ ]:
# Replace this example with the exact path printed above.
REFERENCE_IMAGE_PATH = Path("/kaggle/input/your-private-dataset/reference-photo.jpg")
if not REFERENCE_IMAGE_PATH.is_file():
    raise FileNotFoundError(f"Set REFERENCE_IMAGE_PATH to the attached photo: {REFERENCE_IMAGE_PATH}")

## 3. Add a private Hugging Face token in Kaggle Secrets

The SDXL repository may require accepting its model terms on Hugging Face. If required, accept them on the model page, create a read-only Hugging Face token, then save it in Kaggle's **Add-ons → Secrets** as `HF_TOKEN` and grant this notebook access. The token is read from Kaggle Secrets at runtime and passed only to the cloned script; it is never printed or written into the project files. If Kaggle Secrets is unavailable, stop and ask for help rather than pasting the token into code.

In [ ]:
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Create and grant this notebook access to the Kaggle secret HF_TOKEN.")
print("HF_TOKEN loaded from Kaggle Secrets (value hidden).")

## 4. Clone the tracked repository and run its script

This downloads the current public `main` branch so Kaggle executes the same Python source stored in GitHub. If this fails, check that the notebook and script have been committed and pushed and that the repository is public. Do not copy a separate prompt or model implementation into Kaggle.

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/hann2004/ai_cinematic_storytelling.git"
REPOSITORY_DIR = Path("/kaggle/working/ai_cinematic_storytelling")
if REPOSITORY_DIR.exists():
    if not (REPOSITORY_DIR / ".git").is_dir():
        raise FileExistsError(f"Path exists but is not a Git clone: {REPOSITORY_DIR}")
    remote_url = subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "remote", "get-url", "origin"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if remote_url != REPOSITORY_URL:
        raise RuntimeError(f"Unexpected Git remote at {REPOSITORY_DIR}: {remote_url}")
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

artifact_dir = Path("/kaggle/working/character_reference_artifacts")
artifact_dir.mkdir(exist_ok=True)
environment = os.environ.copy()
environment["HF_TOKEN"] = hf_token
subprocess.run(
    [
        "python",
        "-m",
        "pipeline.kaggle_character_reference",
        "--reference-image",
        str(REFERENCE_IMAGE_PATH),
        "--output-dir",
        str(artifact_dir),
    ],
    cwd=REPOSITORY_DIR,
    env=environment,
    check=True,
)
print("Generation code came from GitHub:", REPOSITORY_URL)

## 5. Review and download the output plus provenance

Review the image before using it. Download the PNG, JSON metadata, and one-line JSONL run record. The metadata/log does not contain the private photo path. Copy the approved image and metadata into this repository's `outputs/`; append the run-record line to the repository's existing `runs.jsonl` rather than replacing the file. The notebook and model code remain the reproducible GitHub source; the private input remains only in Kaggle.

In [ ]:
from IPython.display import FileLink, display
from PIL import Image

image_output = artifact_dir / "character_reference_ipadapter.png"
display(Image.open(image_output))
for filename in (
    "character_reference_ipadapter.png",
    "character_reference_ipadapter.json",
    "character_reference_run.jsonl",
):
    display(FileLink(str(artifact_dir / filename)))